## Executive Summary

**Purpose:** Size the parts around the gears so the whole stage can be made: the sun-to-shaft joint, the planet bearings, the planet pins, the carrier, the ring fixing, and the axial stack that has to fit the envelope length.

**What it does:** `size_components` takes a `PlanetarySet`, its ISO 6336 rating, and a `ComponentInputs` describing the envelope and the input shaft, and returns every component dimension with its check. `rim_thicknesses` gives the rims under the tooth roots (sun over its key groove, planet over its bearing bore, ring outside its roots), which `gears.rating` needs for the rim factor $Y_B$.

**Arrangement assumed:** double-plate (straddle) carrier with $N$ posts between the planets; the output is taken from one plate. Each planet runs on a needle roller cage assembly placed directly on a hardened pin and in the hardened planet bore. The sun is keyed (DIN 6885-1) or splined to the input shaft. The ring gear is the outer wall of the stage and is clamped between housing parts by through bolts, or pressed and pinned into a housing when the rim is too thin for bolt holes.

## How It Fits Into the Gear Design Workflow

**Upstream:** `gears.layout.PlanetarySet` (centre distance, gear diameters), `gears.rating.PlanetaryRating` (torques and tooth forces).

**Downstream:** `gears.manufacturing` (drawing data: fits and tolerances of the pin, bore and carrier holes), `gears.design`.

In [ ]:
#| default_exp gears.components

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass, field

from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import PlanetaryRating

## Sun–Shaft Joint: Theory

**Parallel key (DIN 6885-1, form A).** The key size follows from the shaft diameter. The hub groove depth $t_2$ sets the rim left under the sun teeth. The key transmits the sun torque by flank pressure on the hub side:

$$p = \frac{2000\,K_A T_s}{d_{sh}\,(h - t_1)\,l_{tr}} \le p_{allow},\qquad l_{tr} = l - b_{key}\ \text{(rounded ends)},$$

with $p_{allow} \approx 0.9\,R_e$ of the weaker part for steady load; 150 MPa is a safe default for a hardened sun on a steel shaft. A sun that is case carburized needs its key groove cut before heat treatment (or the groove masked during carburizing), because the hardened case is too hard to broach.

**Involute spline (DIN 5480)** or **an integral pinion shaft** are the alternatives when the key leaves too thin a rim. For a spline the rim is measured from the spline root; take $t_2 \approx 0.55\,m_{spline}$ for a first estimate.

In [ ]:
#| export
# DIN 6885-1 parallel keys: (d_min, d_max] → (b, h, t1 shaft groove, t2 hub groove), mm
DIN6885_KEYS = [
    (6, 8, 2, 2, 1.2, 1.0), (8, 10, 3, 3, 1.8, 1.4), (10, 12, 4, 4, 2.5, 1.8), (12, 17, 5, 5, 3.0, 2.3),
    (17, 22, 6, 6, 3.5, 2.8), (22, 30, 8, 7, 4.0, 3.3), (30, 38, 10, 8, 5.0, 3.3), (38, 44, 12, 8, 5.0, 3.3),
    (44, 50, 14, 9, 5.5, 3.8), (50, 58, 16, 10, 6.0, 4.3), (58, 65, 18, 11, 7.0, 4.4), (65, 75, 20, 12, 7.5, 4.9),
    (75, 85, 22, 14, 9.0, 5.4), (85, 95, 25, 14, 9.0, 5.4), (95, 110, 28, 16, 10.0, 6.4),
]


@dataclass(frozen=True)
class ParallelKey:
    "DIN 6885-1 form A key for one shaft diameter. Lengths in mm."
    d: float
    b: float
    h: float
    t1: float
    t2: float

    @classmethod
    def for_shaft(cls, d: float) -> "ParallelKey":
        for lo, hi, b, h, t1, t2 in DIN6885_KEYS:
            if lo < d <= hi:
                return cls(d, b, h, t1, t2)
        raise ValueError(f"no DIN 6885-1 key listed for d = {d} mm (6 < d ≤ 110)")

    def pressure(self, torque: float, length: float) -> float:
        "Hub-side flank pressure [MPa] for torque [N·m] and key length [mm] (rounded ends)."
        l_tr = length - self.b
        if l_tr <= 0:
            raise ValueError("key shorter than its width")
        return 2000 * torque / (self.d * (self.h - self.t1) * l_tr)

    @staticmethod
    def standard_length(l_max: float) -> float:
        "Longest DIN 6885 standard key length not above l_max [mm]."
        L = [6, 8, 10, 12, 14, 16, 18, 20, 22, 25, 28, 32, 36, 40, 45, 50, 56, 63, 70, 80, 90, 100, 110, 125]
        ok = [x for x in L if x <= l_max + 1e-9]
        if not ok:
            raise ValueError("no standard key length fits")
        return float(ok[-1])

In [ ]:
k14 = ParallelKey.for_shaft(14)
print(k14, f"p = {k14.pressure(10.0, 16):.0f} MPa at 10 N·m, 16 mm key")
assert (k14.b, k14.h, k14.t2) == (5, 5, 2.3) and ParallelKey.standard_length(19) == 18

## Planet Bearing: Theory

**Load.** The sun and ring push on each planet with the same force $F_{bn}$ along their two lines of action. Their tangential parts add and their radial parts nearly cancel (exactly when $\alpha_{w1} = \alpha_{w2}$), so the pin carries

$$F_{pin} = K_\gamma F_{bn}\sqrt{(\cos\alpha_{w1} + \cos\alpha_{w2})^2 + (\sin\alpha_{w1} - \sin\alpha_{w2})^2} + m_p\,\omega_c^2\,a_w,$$

the last term being the planet's centrifugal force, $m_p$ its mass and $\omega_c$ the carrier speed. The bearing turns at the planet's speed relative to the carrier.

**Required capacity** for an $L_{10h}$ basic rating life (roller bearing, exponent 10/3):

$$C_{req} = P\left(\frac{60\,n\,L_{10h}}{10^6}\right)^{3/10}.$$

**Space.** The bearing outer raceway is the planet bore; the rim between the planet root and the bore must keep at least $c_p$ tooth depths ($c_p = 1.2$: no rim penalty). That fixes the largest bore $E_w$. `size_components` starts from an 8 mm bore and takes the smallest whole-mm bore whose bearing reaches the required life and whose pin passes the checks below; a smaller pin keeps the carrier bosses inside the ring. A needle roller cage assembly (ISO 3030, type K $F_w \times E_w \times B_c$) has rollers of diameter $D_w$ with $E_w = F_w + 2D_w$, so the pin diameter is $F_w$.

**Capacity estimate.** Until a catalogue part is chosen, the dynamic load rating of the cage assembly is estimated with the ISO 281 formula for line-contact roller bearings,

$$C_r = b_m f_c (L_{we})^{7/9} Z^{3/4} D_w^{29/27},\qquad b_m = 1.1,$$

with $Z$ rollers at a pitch of $1.25 D_w$ on the pitch circle $D_{pw} = F_w + D_w$, effective roller length $L_{we} = B_c - 3$ mm and $f_c$ from ISO 281 Table 7 as a function of $\gamma = D_w/D_{pw}$. It is within about 20 % of catalogue values; select the real part from the INA, SKF, IKO or NBC catalogue with $C_r \ge C_{req}$ and check its own speed limit.

**Raceway requirements** when rolling directly on pin and bore: 58–64 HRC, case depth ≥ 0.3 mm plus grinding allowance, $R_a \le 0.2\ \mu$m, roundness and cylindricity within IT3–IT4. Normal operating clearance: planet bore **G6**, pin **j5**.

In [ ]:
#| export
_FC_TABLE = [(0.01, 52.1), (0.02, 60.8), (0.03, 66.5), (0.04, 70.7), (0.05, 74.1), (0.06, 76.9), (0.07, 79.2),
             (0.08, 81.2), (0.09, 82.8), (0.10, 84.2), (0.12, 86.4), (0.14, 87.7), (0.16, 88.5), (0.18, 88.8),
             (0.20, 88.7), (0.22, 88.2), (0.24, 87.5), (0.26, 86.4), (0.28, 85.2), (0.30, 83.8)]


def _fc(gamma: float) -> float:
    pts = _FC_TABLE
    if gamma <= pts[0][0]:
        return pts[0][1]
    for (g0, f0), (g1, f1) in zip(pts, pts[1:]):
        if gamma <= g1:
            return f0 + (f1 - f0) * (gamma - g0) / (g1 - g0)
    return pts[-1][1]


def needle_roller_diameter(Fw: float) -> float:
    "Typical roller diameter [mm] of a needle cage assembly for pin diameter Fw [mm]."
    for lim, Dw in ((8, 1.5), (12, 2.0), (25, 2.5), (40, 3.0), (60, 3.5), (1e9, 4.0)):
        if Fw <= lim:
            return Dw
    return 4.0


def needle_cage_rating(Fw: float, Dw: float, Bc: float) -> tuple[float, int]:
    "Estimated dynamic load rating C_r [N] and roller count of a needle cage assembly (ISO 281 formula)."
    Dpw = Fw + Dw
    Z = int(math.pi * Dpw / (1.25 * Dw))
    Lwe = max(Bc - 3.0, 0.6 * Bc)
    return 1.1 * _fc(Dw / Dpw) * Lwe ** (7 / 9) * Z ** 0.75 * Dw ** (29 / 27), Z

In [ ]:
C, Z = needle_cage_rating(20, 3, 17)
print(f"K 20×26×17 estimate: C_r ≈ {C/1000:.1f} kN with {Z} rollers")
assert 15e3 < C < 26e3

## Planet Pin: Theory

The pin is held at both ends in the carrier plates. Treating it as simply supported at mid-plate (span $L = B_c + 2s_w + t_c$, with thrust washers of thickness $s_w$ and plates of thickness $t_c$) and loaded by $F_{pin}$ spread over the bearing length $B_c$ is conservative for press-fitted ends:

$$M_{max} = F_{pin}\left(\frac{L}{4} - \frac{B_c}{8}\right),\qquad \sigma_b = \frac{32\,M_{max}}{\pi d_{pin}^3},\qquad \delta = \frac{F_{pin}\,(8L^3 - 4LB_c^2 + B_c^3)}{384\,E\,I}.$$

The pin load keeps a fixed direction relative to the carrier, so the pin stress is steady and the check is against the hardened pin's yield with a margin (default allowable 400 MPa for 100Cr6 at 60 HRC, which covers stress raisers at the plate edges). Deflection above about 0.01 mm per 10 mm of face width will push $K_{H\beta}$ above the default 1.2.

Pressure of the pin on each plate bore: $p = F_{pin}/(2\,d_{pin}\,t_c)$; keep it under about 150 MPa for a steel carrier. Pin fit in the plates: the pin is ground to one diameter (j5, for the bearing) and the plate holes are **N7**, which gives a light press fit of about 2–30 µm; secure the pin against turning (flat and plate, or a cross pin).

## Carrier: Theory

The plates must hold the pins with a ring of material around each hole, and pass inside the ring's tip circle during assembly:

$$2a_w + d_{pin} + 2e \le D_c \le d_{a,r} - 2c,$$

with $e \ge \max(3\text{ mm}, 0.5\,d_{pin})$ the boss wall and $c$ = 1 mm. Between the planets, $N$ posts tie the two plates together. A post has the tangential width left between neighbouring planet tips at radius $a_w$, $w_{post} = 2a_w\sin(\pi/N) - d_{a,p} - 2c$, and runs radially from the sun tip (plus clearance) to the plate edge. The free plate passes half the planet forces to the output plate through the posts, so each post is a beam fixed at both ends with a shear force

$$V = \frac{T_c}{2N\,r_{post}},\qquad M = \frac{V\,L_{post}}{2},\qquad \sigma = \frac{6M}{h_{post}\,w_{post}^2},$$

$L_{post}$ the clear length between the plates and $r_{post}$ the radius of the post centroid. Keep $\sigma$ under about 120 MPa for a steel carrier (the post sees a pulsating load). The output shaft on the carrier plate needs $d \ge \sqrt[3]{16\,K_A T_c/(\pi\,\tau_{allow})}$, with $\tau_{allow}$ ≈ 40 MPa for a keyed steel shaft.

## Ring Fixing: Theory

The ring reaction torque $T_r = T_c - T_s$ goes into the housing. If $z_b$ bolts clamp the ring axially on a bolt circle $D_{bc}$, friction alone must carry it with a slip safety $S_{slip}$:

$$z_b \ge \frac{2000\,S_{slip}\,K_A\,T_r}{\mu\,D_{bc}\,F_M}$$

with the assembly preload $F_M$ of the bolt (VDI 2230 Table A1, property class 8.8, thread friction 0.12, 90 % yield), $\mu$ = 0.15 between dry steel faces and $S_{slip}$ = 1.5. Bolts through the ring need a rim wide enough for the clearance hole (ISO 273 medium) plus 1.5 mm of wall on each side, and the ring rim inside the bolt holes must still satisfy $s_R \ge 3.5\,m$. When the rim is too thin, press the ring into the housing (H7/s6 class) and pin it with dowels.

## Axial Stack

The length needed is two carrier plates, two thrust washers per planet, the face width, and the running clearances; the sun hub sits inside it. The axial envelope must hold

$$L_{stack} = 2t_c + b + 2s_w + 2c_{ax}.$$

In [ ]:
#| export
# VDI 2230 Table A1 (approximate): 8.8, μ_G = 0.12 → (thread, assembly preload F_M [N], tightening torque M_A [N·m], clearance hole [mm])
BOLTS_88 = {"M3": (2_300, 1.3, 3.4), "M4": (4_000, 3.0, 4.5), "M5": (6_600, 5.9, 5.5), "M6": (9_300, 10.1, 6.6),
            "M8": (17_000, 24.6, 9.0), "M10": (27_000, 48.0, 11.0), "M12": (39_400, 84.0, 13.5)}


@dataclass(frozen=True)
class ComponentInputs:
    """
    Envelope and choices for the parts around the gears. Lengths in mm.

    envelope_d       : outside diameter available [mm]
    envelope_length  : axial length available for the stage [mm] (None: not checked)
    shaft_d          : input shaft diameter in the sun bore [mm]
    hub_joint        : 'key' (DIN 6885-1), 'spline' (DIN 5480) or 'integral' (sun cut on the shaft)
    spline_module    : module of the spline if hub_joint == 'spline'
    housing_wall     : housing wall outside the ring (0: the ring is the outer wall)
    planet_rim       : rim over the planet bore, in tooth depths (1.2: no rim penalty)
    bearing_life_h   : required L10h of the planet bearings (None: same as the gear life)
    plate_t          : carrier plate thickness (None: max(4, 0.6·d_pin))
    washer_t         : thrust washer thickness per side
    axial_clearance  : running clearance per side
    bolt             : bolt size for clamping the ring ('M3' … 'M12')
    p_key_allow, sigma_pin_allow, p_plate_allow, sigma_post_allow, tau_shaft_allow : allowables [MPa]
    """
    envelope_d: float
    shaft_d: float
    envelope_length: float | None = None
    hub_joint: str = "key"
    spline_module: float = 1.0
    housing_wall: float = 0.0
    planet_rim: float = 1.2
    bearing_life_h: float | None = None
    plate_t: float | None = None
    washer_t: float = 1.0
    axial_clearance: float = 0.5
    bolt: str = "M5"
    p_key_allow: float = 150.0
    sigma_pin_allow: float = 400.0
    p_plate_allow: float = 150.0
    sigma_post_allow: float = 120.0
    tau_shaft_allow: float = 40.0


def _hub_depth(ci: ComponentInputs) -> float:
    if ci.hub_joint == "key":
        return ParallelKey.for_shaft(ci.shaft_d).t2
    if ci.hub_joint == "spline":
        return 0.55 * ci.spline_module + 1.1 * ci.spline_module   # spline root below the nominal bore
    return 0.0


def _cage_for_bore(Ew: float) -> tuple[float, float]:
    "Pin diameter F_w and roller diameter D_w for a planet bore E_w [mm]."
    Dw = needle_roller_diameter(Ew - 5)
    Fw = Ew - 2 * Dw
    for _ in range(3):                       # settle the roller size for the pin actually found
        Dw = needle_roller_diameter(Fw)
        Fw = Ew - 2 * Dw
    return Fw, Dw


def max_planet_bore(ps: PlanetarySet, ci: ComponentInputs) -> float:
    "Largest whole-mm planet bore E_w [mm] that keeps the planet rim of `ci.planet_rim` tooth depths."
    p = ps.planet
    return float(math.floor(p.df - 2 * ci.planet_rim * (p.ha + p.hf) * p.m))


def rim_thicknesses(ps: PlanetarySet, ci: ComponentInputs, planet_bore: float | None = None) -> dict[str, float | None]:
    """
    Rim under the tooth roots for the rim factor Y_B [mm]: sun over the hub groove, planet over its
    bore (default: the largest bore allowed), ring outside its roots.
    """
    t2 = _hub_depth(ci)
    sun_rim = None if ci.hub_joint == "integral" else ps.sun.df / 2 - ci.shaft_d / 2 - t2
    Ew = max_planet_bore(ps, ci) if planet_bore is None else planet_bore
    ring_od = ci.envelope_d - 2 * ci.housing_wall
    return {"sun": sun_rim, "planet": (ps.planet.df - Ew) / 2, "ring": (ring_od - ps.ring.df) / 2}



In [ ]:
#| export
@dataclass(frozen=True)
class Check:
    "One component check: value against a limit."
    name: str
    value: float
    limit: float
    unit: str
    upper: bool = True          # True: value must not exceed limit

    @property
    def ok(self) -> bool:
        return self.value <= self.limit + 1e-9 if self.upper else self.value >= self.limit - 1e-9

    def __str__(self) -> str:
        rel = "≤" if self.upper else "≥"
        return f"{'✓' if self.ok else '✗'} {self.name}: {self.value:.3g} {self.unit} (need {rel} {self.limit:.3g})"


@dataclass(frozen=True)
class PlanetaryComponents:
    """
    Dimensions of the parts around the gears (mm, N, N·m) and their checks.

    sun_hub    : dict  joint type, key size and length or spline data, rim
    bearing    : dict  cage assembly Fw × Ew × Bc, rollers, load, speed, C_req, C_est, L10h
    pin        : dict  diameter, length, span, stress, deflection, fits
    carrier    : dict  plate diameter and thickness, pin circle, posts, output shaft
    ring_mount : dict  ring OD, bolts or press fit, torque
    axial      : dict  stack length and its parts
    lubrication: str   starting recommendation
    checks     : tuple[Check, ...]
    """
    sun_hub: dict
    bearing: dict
    pin: dict
    carrier: dict
    ring_mount: dict
    axial: dict
    lubrication: str
    checks: tuple[Check, ...]

    @property
    def ok(self) -> bool:
        return all(c.ok for c in self.checks)

    def problems(self) -> list[str]:
        return [str(c) for c in self.checks if not c.ok]


def lubrication_advice(v: float) -> str:
    "Starting lubricant choice from the pitch line velocity v [m/s] (AGMA 9005 style ranges)."
    if v < 1.5:
        return f"v = {v:.1f} m/s: semi-fluid grease NLGI 00 (EP, synthetic base) or ISO VG 320 oil"
    if v < 5:
        return f"v = {v:.1f} m/s: ISO VG 220 EP gear oil, splash lubrication, or NLGI 00 grease for sealed-for-life units"
    if v < 12:
        return f"v = {v:.1f} m/s: ISO VG 150 EP gear oil, splash lubrication; grease not recommended"
    return f"v = {v:.1f} m/s: ISO VG 100 EP gear oil, spray or forced lubrication"


def size_components(ps: PlanetarySet, rating: PlanetaryRating, ci: ComponentInputs,
                    life_h: float, n_in: float, K_A: float = 1.25, density: float = 7.85e-6) -> PlanetaryComponents:
    """
    Size the sun joint, planet bearings, pins, carrier and ring fixing (see the theory sections).

    life_h, n_in, K_A: as used for the rating; density in kg/mm³ for the planet centrifugal force.
    """
    checks: list[Check] = []
    sun, pl, ring = ps.sun, ps.planet, ps.ring
    N = ps.n_planets
    rims = rim_thicknesses(ps, ci)
    sp = ps.speeds(n_in)
    if rating.factors.get("K_gamma") is None:
        raise ValueError("rating has no K_gamma")

    # --- sun hub -----------------------------------------------------------------
    hub: dict = {"joint": ci.hub_joint, "shaft_d": ci.shaft_d, "bore_fit": "H7 (shaft k6)", "rim": rims["sun"]}
    if ci.hub_joint == "key":
        key = ParallelKey.for_shaft(ci.shaft_d)
        L_key = ParallelKey.standard_length(sun.b + 2 * ci.washer_t)
        p_key = key.pressure(K_A * rating.T_sun, L_key)
        hub.update(key=f"DIN 6885-1 A {key.b:g} × {key.h:g} × {L_key:g}", key_b=key.b, key_h=key.h,
                   t1=key.t1, t2=key.t2, key_length=L_key, key_pressure=p_key,
                   hub_groove_width="P9 (fixed key) or JS9", hub_groove_depth=f"{ci.shaft_d + key.t2:.1f} +0.1/0 over bore")
        checks.append(Check("key flank pressure", p_key, ci.p_key_allow, "MPa"))
    elif ci.hub_joint == "spline":
        hub.update(spline=f"DIN 5480 W {ci.shaft_d:g} × {ci.spline_module:g} (choose z from the standard)")
    if rims["sun"] is not None:
        checks.append(Check("sun rim over hub groove / tooth depth", rims["sun"] / ((sun.ha + sun.hf) * sun.m), 0.5, "", upper=False))

    # --- planet bearing and pin: smallest cage assembly that passes ---------------------
    Bc = math.floor(pl.b - 1.0) if pl.b > 6 else pl.b
    a1, a2 = ps.sun_planet.alpha_w, ps.planet_ring.alpha_w
    Kg = rating.factors["K_gamma"]
    F_teeth = Kg * rating.F_bn * math.hypot(math.cos(a1) + math.cos(a2), math.sin(a1) - math.sin(a2))
    w_c = 2 * math.pi * sp["carrier"] / 60
    n_b = abs(sp["planet_rel"])
    L_h = ci.bearing_life_h or life_h
    E = 210_000.0
    Ew_max = max_planet_bore(ps, ci)
    trial = None
    for Ew in [float(x) for x in range(8, int(Ew_max) + 1)] or [Ew_max]:
        Fw, Dw = _cage_for_bore(Ew)
        if Fw < 4:
            continue
        m_p = density * math.pi / 4 * (pl.d ** 2 - Ew ** 2) * pl.b              # kg
        F_cf = m_p * w_c ** 2 * ps.a_w / 1000                                   # N
        F_pin = F_teeth + F_cf
        C_est, Z = needle_cage_rating(Fw, Dw, Bc)
        L10h = (C_est / F_pin) ** (10 / 3) * 1e6 / (60 * n_b) if n_b > 0 else math.inf
        t_c = ci.plate_t if ci.plate_t is not None else max(4.0, math.ceil(0.5 * Fw))
        L = pl.b + 2 * ci.washer_t + t_c
        M = F_pin * (L / 4 - Bc / 8)
        sig_pin = 32 * M / (math.pi * Fw ** 3)
        I = math.pi * Fw ** 4 / 64
        delta = F_pin * (8 * L ** 3 - 4 * L * Bc ** 2 + Bc ** 3) / (384 * E * I)
        p_plate = F_pin / (2 * Fw * t_c)
        trial = (Ew, Fw, Dw, Z, F_pin, F_cf, C_est, L10h, t_c, L, M, sig_pin, delta, p_plate)
        if (L10h >= L_h and sig_pin <= ci.sigma_pin_allow and delta / pl.b * 10 <= 0.01
                and p_plate <= ci.p_plate_allow):
            break
    if trial is None:
        raise ValueError("planet is too small for any needle cage assembly")
    Ew, Fw, Dw, Z, F_pin, F_cf, C_est, L10h, t_c, L, M, sig_pin, delta, p_plate = trial
    C_req = F_pin * (60 * n_b * L_h / 1e6) ** 0.3
    cage_txt = f"needle roller cage assembly K {Fw:g} × {Ew:g} × {Bc:g} (ISO 3030), or equivalent"
    if Bc > 2 * Fw:
        cage_txt += f"; if no single cage this wide is listed, use two cages of about {Bc / 2:.0f} mm side by side"
    brg = {"type": cage_txt,
           "Fw": Fw, "Ew": Ew, "Bc": Bc, "Dw": Dw, "rollers": Z, "F_pin": F_pin, "F_centrifugal": F_cf,
           "speed_rpm": n_b, "C_req": C_req, "C_est": C_est, "L10h_est": L10h, "planet_rim": (pl.df - Ew) / 2,
           "raceways": "pin and planet bore 58–64 HRC, Ra ≤ 0.2 µm, cylindricity IT3", "bore_fit": "G6", "pin_fit": "j5"}
    checks.append(Check("planet bearing L10h (estimated rating)", L10h, L_h, "h", upper=False))
    pin_len = pl.b + 2 * ci.washer_t + 2 * t_c
    pin = {"d": Fw, "length": pin_len, "span": L, "M_max": M, "sigma_b": sig_pin, "deflection": delta,
           "plate_pressure": p_plate, "material": "100Cr6 (1.3505) through hardened 58–64 HRC, or 16MnCr5 case hardened",
           "fit_in_plates": "plate holes N7 on the j5 pin (light press), pin secured against rotation", "raceway_fit": "j5", "surface": "Ra ≤ 0.2 µm on raceway"}
    checks += [Check("pin bending stress", sig_pin, ci.sigma_pin_allow, "MPa"),
               Check("pin pressure on plate bore", p_plate, ci.p_plate_allow, "MPa"),
               Check("pin deflection per 10 mm face", delta / pl.b * 10, 0.01, "mm")]

    # --- carrier ----------------------------------------------------------------------
    e = max(3.0, 0.5 * Fw)
    Dc_min = 2 * ps.a_w + Fw + 2 * e
    Dc_max = ring.da - 2.0
    Dc = math.floor(Dc_max) if Dc_max >= Dc_min else Dc_min
    w_post = 2 * ps.a_w * math.sin(math.pi / N) - pl.da - 2.0
    r_in = sun.da / 2 + 1.0
    h_post = Dc / 2 - r_in
    r_post = (Dc / 2 + r_in) / 2
    L_post = pl.b + 2 * ci.washer_t
    T_c = rating.T_carrier
    sig_post = math.inf
    if w_post > 0 and h_post > 0:
        V = K_A * T_c * 1000 / (2 * N * r_post)
        sig_post = 6 * (V * L_post / 2) / (h_post * w_post ** 2)
    d_out = (16 * K_A * T_c * 1000 / (math.pi * ci.tau_shaft_allow)) ** (1 / 3)
    carrier = {"type": "double-plate carrier with posts, output from one plate",
               "plate_d": Dc, "plate_t": t_c, "pin_circle_d": 2 * ps.a_w, "pin_holes": f"{N} × Ø{Fw:g} N7 at {360 / N:g}°",
               "post_width": w_post, "post_height": h_post, "post_length": L_post, "post_stress": sig_post,
               "sun_clearance_bore": 2 * r_in, "output_shaft_min_d": d_out,
               "material": "42CrMo4 Q&T (1.7225) or C45 normalized, or sintered steel for series",
               "pin_hole_position": "pin circle radius ±(f_a) and angular position, see manufacturing data"}
    checks += [Check("carrier plate inside ring tip circle", Dc_min, Dc_max, "mm"),
               Check("carrier post width between planets", w_post, 3.0, "mm", upper=False),
               Check("carrier post bending stress", sig_post, ci.sigma_post_allow, "MPa")]

    # --- ring fixing --------------------------------------------------------------------
    ring_od = ci.envelope_d - 2 * ci.housing_wall
    F_M, M_A, d_h = BOLTS_88[ci.bolt]
    T_r = abs(rating.T_ring)
    rim_w = (ring_od - ring.df) / 2
    D_bc = (ring_od + ring.df) / 2 + (d_h / 2 if rim_w > 0 else 0) * 0
    need_rim = d_h + 2 * 1.5 + 3.5 * ring.m
    if rim_w >= need_rim:
        D_bc = ring_od - 2 * (1.5 + d_h / 2)
        z_b = max(N, math.ceil(2000 * 1.5 * K_A * T_r / (0.15 * D_bc * F_M)))
        mount = {"method": f"{z_b} × {ci.bolt}-8.8 through bolts on Ø{D_bc:.1f} bolt circle, clamped between housing parts",
                 "bolts": z_b, "bolt_circle_d": D_bc, "tightening_torque": M_A, "clearance_hole": d_h,
                 "dowels": "2 × ISO 8734 dowel pins for location"}
    else:
        mount = {"method": "ring pressed into housing (H7/s6) and secured with dowel pins; rim too thin for bolt holes",
                 "bolts": 0}
    mount.update(ring_od=ring_od, ring_rim=rim_w, ring_torque=T_r)
    checks.append(Check("ring rim / module (ISO 6336-3 lower limit; Y_B > 1 below 3.5)", rim_w / ring.m, 1.75, "", upper=False))
    if ci.housing_wall == 0 and ring_od > ci.envelope_d + 1e-9:
        checks.append(Check("ring OD within envelope", ring_od, ci.envelope_d, "mm"))

    # --- axial stack ----------------------------------------------------------------------
    stack = 2 * t_c + pl.b + 2 * ci.washer_t + 2 * ci.axial_clearance
    axial = {"plates": 2 * t_c, "face_width": pl.b, "washers": 2 * ci.washer_t,
             "clearances": 2 * ci.axial_clearance, "stack": stack,
             "thrust_washers": f"2 per planet, Ø{Fw + 0.5:.1f}/Ø{min(pl.df - 1, Ew + 6):.1f} × {ci.washer_t:g}, hardened steel or PEEK"}
    if ci.envelope_length is not None:
        checks.append(Check("axial stack within envelope length", stack, ci.envelope_length, "mm"))

    return PlanetaryComponents(hub, brg, pin, carrier, mount, axial, lubrication_advice(rating.v), tuple(checks))

### Example

The rated set from `gears.rating` (18/27/72, module 1.5, 20 mm face) in a 120 mm envelope on a 14 mm keyed shaft:

In [ ]:
from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import RatingInputs, rate_planetary
ps = PlanetarySet.build(18, 27, 72, m=1.5, n_planets=3, b=20)
ci = ComponentInputs(envelope_d=120, shaft_d=14, envelope_length=40)
inp = RatingInputs(torque_out=40.0, n_in=3000.0)
r0 = rate_planetary(ps, inp, rims=rim_thicknesses(ps, ci))              # pass 1: largest planet bore
comp0 = size_components(ps, r0, ci, life_h=inp.life_h, n_in=inp.n_in)
rims = rim_thicknesses(ps, ci, planet_bore=comp0.bearing["Ew"])          # pass 2: the bore actually chosen
print({k: (round(v, 2) if v is not None else None) for k, v in rims.items()})
r = rate_planetary(ps, inp, rims=rims)
comp = size_components(ps, r, ci, life_h=inp.life_h, n_in=inp.n_in)
for name in ("sun_hub", "bearing", "pin", "carrier", "ring_mount", "axial"):
    d = getattr(comp, name)
    print(name + ":", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items()})
print(comp.lubrication)
for c in comp.checks:
    print(c)

The stage is 51 mm long against a 40 mm envelope, so this face width does not fit; `gears.design` searches for a set that does. The rating is run twice because the planet rim depends on the bearing, which depends on the load. The sun rim over the key groove is thin here, so the sun's bending stress carries a rim factor above 1:

In [ ]:
sun_r = next(g for g in r.gears if g.gear == "sun")
print(f"sun Y_B = {sun_r.Y_B:.3f}, S_F = {sun_r.S_F:.2f}")
assert sun_r.Y_B > 1.0

---

## Tests

In [ ]:
#| hide
# pin load: twice the line-of-action force when both meshes have the same α_w (ignoring centrifugal force)
assert math.isclose(comp.bearing["F_pin"] - comp.bearing["F_centrifugal"], 2 * r.F_bn * math.cos(ps.sun_planet.alpha_w) * r.factors["K_gamma"], rel_tol=1e-9)
# bearing geometry
b = comp.bearing
assert math.isclose(b["Ew"], b["Fw"] + 2 * b["Dw"]) and b["Ew"] <= ps.planet.df - 2 * 1.2 * 2.25 * 1.5
assert b["L10h_est"] >= 10_000
# carrier plate fits inside the ring tip circle and holds the pins
assert comp.carrier["plate_d"] <= ps.ring.da - 2 and comp.carrier["plate_d"] >= 2 * ps.a_w + b["Fw"]
# more torque needs at least as large a bearing
r_hi = rate_planetary(ps, RatingInputs(torque_out=200.0, n_in=3000.0), rims=rims)
comp_hi = size_components(ps, r_hi, ci, life_h=10_000, n_in=3000.0)
assert comp_hi.bearing["F_pin"] > comp.bearing["F_pin"] and comp_hi.bearing["Ew"] >= comp.bearing["Ew"]
# keys: every listed shaft diameter maps to a key
for d in (7, 9, 11, 14, 20, 25, 35, 40, 48, 55, 60, 70, 80, 90, 100):
    ParallelKey.for_shaft(d)